# Oracle

In [ ]:

import csv
import math
import re
from pathlib import Path
from collections import defaultdict

# ===== Paths (H100 only) =====
SPEC_ROOT_DIR = Path("/home/ac.zzheng/power/GPGPU/data/H100/spec_power_motif")
ML_ROOT_DIR   = Path("/home/ac.zzheng/power/GPGPU/data/H100/ml_power_motif")

SPEC_APPS = None   # None -> auto-discover
ML_APPS = None     # None -> auto-discover

PROFILE_SECONDS = 10.0
ACTIVE_POWER_TH = 120.0
ACTIVE_SM_TH = 400.0

FILE_RE = re.compile(r"(?P<cap>\d+)_(?P<gpus>\d+)_gpu_metrics\.csv$")


def f(x):
    try:
        return float(str(x).strip())
    except Exception:
        return None


def mean(xs):
    return sum(xs) / len(xs) if xs else float("nan")


def fmt2(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "nan"
    return f"{x:.2f}"


def list_apps(root, apps):
    if apps is not None:
        return apps
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])


def detect_metric_keys(app_dir):
    keys = set()
    for p in app_dir.iterdir():
        m = FILE_RE.match(p.name)
        if m:
            keys.add((int(m.group("cap")), int(m.group("gpus"))))
    return keys


def read_runtime_lookup_spec(app_dir):
    """
    Build lookup {(total_cap, gpu_count): runtime_seconds}
    Auto-detect whether runtime.csv power_cap is total-cap or per-gpu-cap by overlap with metric keys.
    """
    out = {}
    p = app_dir / "runtime.csv"
    if not p.exists():
        return out

    metric_keys = detect_metric_keys(app_dir)
    rows = []

    with p.open(newline="") as fh:
        r = csv.DictReader(fh)
        cols = set(r.fieldnames or [])
        req = {"power_cap", "gpu_count", "runtime_seconds"}
        if not req.issubset(cols):
            return out
        for row in r:
            cap = f(row.get("power_cap"))
            g = f(row.get("gpu_count"))
            v = f(row.get("runtime_seconds"))
            if cap is None or g is None or v is None:
                continue
            g = int(round(g))
            rows.append((cap, g, v))

    # candidate A: cap already total
    raw = {(int(round(cap)), g): v for cap, g, v in rows}
    # candidate B: cap is per-gpu, convert to total
    per = {(int(round(cap * g)), g): v for cap, g, v in rows}

    raw_hits = sum(1 for k in raw if k in metric_keys)
    per_hits = sum(1 for k in per if k in metric_keys)

    return raw if raw_hits >= per_hits else per


def read_throughput_lookup_ml(app_dir):
    """
    Build lookup {(total_cap, gpu_count): throughput}
    Supports throughput.csv columns:
      - total_gpu_cap (preferred)
      - power_cap (fallback)
      - throughput_images_per_sec / throughput_tokens_per_sec / throughput
    """
    out = {}
    p = app_dir / "throughput.csv"
    if not p.exists():
        return out

    with p.open(newline="") as fh:
        r = csv.DictReader(fh)
        cols = set(r.fieldnames or [])

        # cap column
        cap_col = None
        for c in ("total_gpu_cap", "power_cap"):
            if c in cols:
                cap_col = c
                break
        if cap_col is None or "gpu_count" not in cols:
            return out

        # throughput column
        perf_col = None
        for c in ("throughput_images_per_sec", "throughput_tokens_per_sec", "throughput"):
            if c in cols:
                perf_col = c
                break
        if perf_col is None:
            return out

        has_status = "status" in cols
        for row in r:
            if has_status and str(row.get("status", "")).strip().lower() != "ok":
                continue
            cap = f(row.get(cap_col))
            g = f(row.get("gpu_count"))
            v = f(row.get(perf_col))
            if cap is None or g is None or v is None:
                continue
            out[(int(round(cap)), int(round(g)))] = v

    return out


def extract_metrics(metric_csv):
    m = FILE_RE.match(metric_csv.name)
    if not m:
        return None
    cap = int(m.group("cap"))
    gcount = int(m.group("gpus"))

    with metric_csv.open(newline="") as fh:
        r = csv.DictReader(fh)
        rows = list(r)
        cols = r.fieldnames or []

    if not rows or "Time (s)" not in cols:
        return None

    # Drop idle samples (if available)
    if "GPU0_DRAM_Active" in cols:
        rows = [row for row in rows if (f(row.get("GPU0_DRAM_Active")) is not None and f(row.get("GPU0_DRAM_Active")) != 0.0)]
    if not rows:
        return None

    def colvals(rows_in, c):
        out = []
        for row in rows_in:
            v = f(row.get(c))
            if v is not None:
                out.append(v)
        return out

    times = colvals(rows, "Time (s)")
    if not times:
        return None

    # First PROFILE_SECONDS
    t0 = min(times)
    t1 = t0 + PROFILE_SECONDS
    prof = [row for row in rows if (f(row.get("Time (s)")) is not None and f(row.get("Time (s)")) <= t1)]
    if not prof:
        prof = rows

    gpu_ids = sorted({
        int(c.split("_")[0].replace("GPU", ""))
        for c in cols if c.startswith("GPU") and "_" in c
    })

    if len(gpu_ids) >= gcount:
        gpu_ids = gpu_ids[:gcount]

    sm_avg, dr_avg = {}, {}
    fp16_avg, fp32_avg, fp64_avg = {}, {}, {}

    for gid in gpu_ids:
        sm_avg[gid]   = mean(colvals(prof, f"GPU{gid}_SM_Clock (MHz)"))
        dr_avg[gid]   = mean(colvals(prof, f"GPU{gid}_DRAM_Active"))
        fp16_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP16_Active"))
        fp32_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP32_Active"))
        fp64_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP64_Active"))

    if not gpu_ids:
        return None

    # Sum across all detected GPUs in this file
    dram_active_all = sum([0.0 if math.isnan(dr_avg[g]) else dr_avg[g] for g in gpu_ids])
    sm_active_all   = sum([0.0 if math.isnan(sm_avg[g]) else sm_avg[g] for g in gpu_ids])

    fp16_all = sum([0.0 if math.isnan(fp16_avg[g]) else fp16_avg[g] for g in gpu_ids])
    fp32_all = sum([0.0 if math.isnan(fp32_avg[g]) else fp32_avg[g] for g in gpu_ids])
    fp64_all = sum([0.0 if math.isnan(fp64_avg[g]) else fp64_avg[g] for g in gpu_ids])
    fp_all = fp16_all + fp32_all + fp64_all

    return {
        "power_cap": cap,
        "gpu_count": gcount,
        "dram_active": dram_active_all,
        "sm_active": sm_active_all,
        "fp_all": fp_all,
    }


def run_suite(root, apps, suite_name, kind):
    apps = list_apps(root, apps)
    if not apps:
        print(f"[skip] no apps in {root}")
        return

    for app in apps:
        app_dir = root / app
        if not app_dir.exists():
            continue

        if kind == "runtime":
            target_lookup = read_runtime_lookup_spec(app_dir)
            better = "lower"
        else:
            target_lookup = read_throughput_lookup_ml(app_dir)
            better = "higher"

        by_cap = defaultdict(list)

        for p in sorted(app_dir.iterdir()):
            m = FILE_RE.match(p.name)
            if not m:
                continue

            x = extract_metrics(p)
            if x is None:
                continue

            key = (x["power_cap"], x["gpu_count"])
            target = target_lookup.get(key, None)
            if target is None:
                continue

            x["performance"] = target
            by_cap[x["power_cap"]].append(x)

        if not by_cap:
            continue

        print(f"\n===== {suite_name}/{app} =====")
        print(f"(performance better when {better})")

        for cap in sorted(by_cap.keys()):
            rows = by_cap[cap]
            if kind == "runtime":
                rank_rows = sorted(rows, key=lambda r: r["performance"])  # lower runtime is better
            else:
                rank_rows = sorted(rows, key=lambda r: r["performance"], reverse=True)  # higher throughput is better

            true_rank = [r["gpu_count"] for r in rank_rows]
            print(f"\ncap={cap}\ttrue_rank={true_rank}")
            print("gpu_count\tperformance\tdram_active\tsm_active\tfp_all")
            for r in rank_rows:
                print(
                    f"{r['gpu_count']}\t{fmt2(r['performance'])}\t"
                    f"{fmt2(r['dram_active'])}\t{fmt2(r['sm_active'])}\t{fmt2(r['fp_all'])}"
                )


run_suite(SPEC_ROOT_DIR, SPEC_APPS, "SPEC", "runtime")
run_suite(ML_ROOT_DIR, ML_APPS, "ML", "throughput")

# Unsupervised Pair Ranking

In [5]:
# Unsupervised per-case pairwise ranking (SPEC only)
# - Train separately for each (app, cap)
# - Seed weights -> pseudo labels within that case
# - Fine-tune weights with seed-anchored regularization
# - Evaluate top-1 vs true optimal (runtime lowest)

import re
import numpy as np
from pathlib import Path
from collections import defaultdict

TXT_PATH = Path("/home/ac.zzheng/power/GPGPU/data/H100/h100_rank_results.txt")

SEED_W = np.array([0.75, 0.10, 0.15], dtype=float)  # [dram, sm, fp]
SEED_B = 0.0

LR = 0.03
EPOCHS = 800
L2_SEED = 0.10  # stronger => learned weights stay closer to SEED_W


def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def parse_txt(path):
    lines = path.read_text().splitlines()
    rows = []
    suite = app = better = None
    cap = None
    in_table = False

    for ln in lines:
        if ln.startswith("===== ") and ln.endswith(" ====="):
            m = re.match(r"^=====\s+(\w+)/(.*?)\s+=====$", ln)
            if m:
                suite, app = m.group(1), m.group(2)
            continue

        if ln.startswith("(performance better when "):
            better = "lower" if "lower" in ln else "higher"
            continue

        m = re.match(r"^cap=(\d+)\s+true_rank=\[(.*?)\]$", ln)
        if m:
            cap = int(m.group(1))
            in_table = False
            continue

        if ln.startswith("gpu_count\tperformance\tdram_active\tsm_active\tfp_all"):
            in_table = True
            continue

        if in_table and ln.strip():
            ps = ln.split("\t")
            if len(ps) == 5 and ps[0].isdigit():
                rows.append({
                    "suite": suite,
                    "app": app,
                    "cap": cap,
                    "gpu": int(ps[0]),
                    "perf": float(ps[1]),
                    "dram": float(ps[2]),
                    "sm": float(ps[3]),
                    "fp": float(ps[4]),
                    "better": better,
                })
            else:
                in_table = False

    by_case = defaultdict(list)
    for r in rows:
        by_case[(r["suite"], r["app"], r["cap"])].append(r)
    return by_case


def minmax_norm(vals):
    lo, hi = min(vals), max(vals)
    if hi - lo < 1e-12:
        return [0.0] * len(vals)
    return [(v - lo) / (hi - lo) for v in vals]


def build_case_feats(rows):
    # rows are one case (same app/cap)
    nd = minmax_norm([r["dram"] for r in rows])
    ns = minmax_norm([r["sm"] for r in rows])
    nf = minmax_norm([r["fp"] for r in rows])

    feats = []
    for i, r in enumerate(rows):
        feats.append({
            "gpu": r["gpu"],
            "perf": r["perf"],
            "feat": np.array([nd[i], ns[i], nf[i]], dtype=float),
        })
    return feats


def build_pseudo_pairs(feats, seed_w, seed_b):
    X, y = [], []
    seed_scores = [float(seed_w @ x["feat"] + seed_b) for x in feats]

    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            diff = feats[i]["feat"] - feats[j]["feat"]
            yi = 1.0 if seed_scores[i] > seed_scores[j] else 0.0
            X.append(diff);   y.append(yi)
            X.append(-diff);  y.append(1.0 - yi)

    return np.array(X, dtype=float), np.array(y, dtype=float)


def train_case_unsup(X, y, w0, b0=0.0, lr=0.03, epochs=800, l2_seed=0.10):
    # Very small cases (2 GPUs) can be unstable; anchored regularization helps.
    w = np.array(w0, dtype=float).copy()
    b = float(b0)
    n = len(y)

    if n == 0:
        return w, b

    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        grad_w = (X.T @ (p - y)) / n + l2_seed * (w - w0)  # anchor to seed
        grad_b = np.mean(p - y)
        w -= lr * grad_w
        b -= lr * grad_b

    return w, b


def rank_case(feats, w, b):
    scores = np.zeros(len(feats), dtype=float)
    for i in range(len(feats)):
        for j in range(len(feats)):
            if i == j:
                continue
            scores[i] += sigmoid((feats[i]["feat"] - feats[j]["feat"]) @ w + b)

    order = np.argsort(-scores)
    pred_rank = [feats[k]["gpu"] for k in order]
    return pred_rank


def true_rank_spec(feats):
    # SPEC: lower runtime is better
    order = sorted(range(len(feats)), key=lambda i: feats[i]["perf"])
    return [feats[i]["gpu"] for i in order]


def top1_true_set_spec(feats, tie_thresh=0.05):
    """
    Returns a set of acceptable true top-1 GPUs.
    If top1 and top2 runtimes are within tie_thresh (3%), both are accepted.
    """
    if not feats:
        return set()

    ranked = sorted(feats, key=lambda x: x["perf"])  # lower runtime is better
    best_gpu = ranked[0]["gpu"]

    if len(ranked) < 2:
        return {best_gpu}

    best = ranked[0]["perf"]
    second = ranked[1]["perf"]

    if best <= 0:
        return {best_gpu}

    rel_gap = (second - best) / best
    if rel_gap <= tie_thresh:
        return {ranked[0]["gpu"], ranked[1]["gpu"]}
    return {best_gpu}


# ---------- main ----------
by_case = parse_txt(TXT_PATH)

spec_cases = [(k, v) for k, v in sorted(by_case.items()) if k[0] == "SPEC"]
print("SPEC cases:", len(spec_cases))

hits = 0
results = []
weights = []

for (suite, app, cap), rows in spec_cases:
    feats = build_case_feats(rows)
    X, y = build_pseudo_pairs(feats, SEED_W, SEED_B)

    w_case, b_case = train_case_unsup(
        X, y,
        w0=SEED_W,
        b0=SEED_B,
        lr=LR,
        epochs=EPOCHS,
        l2_seed=L2_SEED
    )

    pred = rank_case(feats, w_case, b_case)
    true = true_rank_spec(feats)
    true_top1_set = top1_true_set_spec(feats, tie_thresh=0.03)
    hit = int(pred[0] in true_top1_set)

    hits += hit
    results.append((app, cap, pred, true, sorted(true_top1_set), hit))
    weights.append(w_case)

acc = hits / len(spec_cases) if spec_cases else 0.0
print(f"Top-1 accuracy (SPEC): {acc:.4f} ({hits}/{len(spec_cases)})")

W = np.array(weights)
print("\nAverage learned per-case weights:")
print("w_dram =", round(float(W[:, 0].mean()), 6))
print("w_sm   =", round(float(W[:, 1].mean()), 6))
print("w_fp   =", round(float(W[:, 2].mean()), 6))

print("\nSample case predictions:")
for app, cap, pred, true, true_top1_set, hit in results[:]:
    print(f"{app:12s} cap={cap:4d} pred={pred} true={true} true_top1_set={true_top1_set} hit={hit}")

print("\nMispredictions:")
for app, cap, pred, true, true_top1_set, hit in results:
    if not hit:
        print(f"{app:12s} cap={cap:4d} pred={pred[0]} true_top1_set={true_top1_set} full_pred={pred} full_true={true}")



SPEC cases: 90
Top-1 accuracy (SPEC): 0.9667 (87/90)

Average learned per-case weights:
w_dram = 1.4714
w_sm   = 0.597337
w_fp   = 0.69497

Sample case predictions:
cloverleaf   cap= 400 pred=[1, 2] true=[1, 2] true_top1_set=[1] hit=1
cloverleaf   cap= 500 pred=[2, 1] true=[2, 1] true_top1_set=[2] hit=1
cloverleaf   cap= 600 pred=[2, 3, 1] true=[2, 3, 1] true_top1_set=[2] hit=1
cloverleaf   cap= 700 pred=[2, 3, 1] true=[2, 3, 1] true_top1_set=[2] hit=1
cloverleaf   cap= 800 pred=[3, 2, 4, 1] true=[3, 2, 4, 1] true_top1_set=[2, 3] hit=1
cloverleaf   cap= 900 pred=[4, 3, 2, 1] true=[3, 4, 2, 1] true_top1_set=[3] hit=0
cloverleaf   cap=1000 pred=[4, 3, 2, 1] true=[4, 3, 2, 1] true_top1_set=[3, 4] hit=1
cloverleaf   cap=1100 pred=[4, 3, 2, 1] true=[4, 3, 2, 1] true_top1_set=[4] hit=1
cloverleaf   cap=1200 pred=[4, 3, 2, 1] true=[4, 3, 2, 1] true_top1_set=[4] hit=1
cloverleaf   cap=1300 pred=[4, 3, 2, 1] true=[4, 3, 2, 1] true_top1_set=[4] hit=1
cloverleaf   cap=1400 pred=[4, 3, 2, 1] true=